# 使用向量嵌入、OpenAI 和 Astra DB 探索哲学

### AstraPy 版本

在本快速入门中，你将学习如何使用 OpenAI 的向量嵌入，并以 DataStax [Astra DB](https://docs.datastax.com/en/astra/home/astra.html) 作为数据持久化的向量存储，构建一个“哲学名言查找器与生成器”。

本 notebook 的基本工作流概述如下。你将计算并存储一批著名哲学家名言的向量嵌入，用它们构建一个强大的搜索引擎，之后甚至还能构建一个新名言生成器！

本 notebook 示范了向量搜索的一些标准使用模式，同时展示了上手 [Astra DB](https://docs.datastax.com/en/astra/home/astra.html) 有多么容易。

关于使用向量搜索和文本嵌入构建问答系统的背景知识，请参阅这个出色的实践 notebook：[使用嵌入进行问答](https://github.com/openai/openai-cookbook/blob/main/examples/Question_answering_using_embeddings.ipynb)。

目录：
- 设置
- 创建向量集合
- 连接 OpenAI
- 将名言加载到向量存储中
- 用例 1：**名言搜索引擎**
- 用例 2：**名言生成器**
- 清理

### 工作原理

**索引**

每条名言都会通过 OpenAI 的 `Embedding` 转换为一个嵌入向量。这些向量会保存在向量存储中，供之后搜索使用。同时还会存储一些元数据，包括作者姓名和其他几个预先计算的标签，以便定制搜索。

![1_vector_indexing](https://user-images.githubusercontent.com/14221764/282422016-1d540607-eed4-4240-9c3d-22ee3a3bc90f.png)

**搜索**

为了找到与给定搜索名言相似的名言，会即时将后者转换为嵌入向量，并用这个向量在存储中查询相似的向量……也就是之前已被索引的相似名言。搜索还可以选择通过额外的元数据进行限定（“帮我找斯宾诺莎说过的、与这句相似的名言……”）。

![2_vector_search](https://user-images.githubusercontent.com/14221764/282422033-0a1297c4-63bb-4e04-b120-dfd98dc1a689.png)

这里的关键在于，“内容相似的名言”在向量空间中对应的是在度量上彼此接近的向量：因此，向量相似度搜索实际上实现了语义相似度。_这正是向量嵌入如此强大的关键原因。_

下面的示意图试图传达这个思想。每条名言转换为向量后，就是空间中的一个点。确切地说，在这里它位于一个球面上，因为与大多数嵌入一样，OpenAI 的嵌入向量都被归一化为_单位长度_。哦，而且这个球面其实不是三维的，而是 1536 维的！

所以本质上，向量空间中的相似度搜索返回的是与查询向量最接近的那些向量：

![3_vector_space](https://user-images.githubusercontent.com/14221764/262321363-c8c625c1-8be9-450e-8c68-b1ed518f990d.png)

**生成**

给定一个建议（一个主题或一条草拟的名言）后，先执行搜索步骤，然后将返回的前几条结果（名言）送入 LLM 提示词，要求生成模型参照传入的示例_以及_最初的建议，创作一段新的文本。

![4_quote_generation](https://user-images.githubusercontent.com/14221764/282422050-2e209ff5-07d6-41ac-99ac-f442e090b3bb.png)

## 设置

安装并导入必要的依赖：

In [1]:
!pip install --quiet "astrapy>=0.6.0" "openai>=1.0.0" datasets

In [2]:
from getpass import getpass
from collections import Counter

from astrapy.db import AstraDB
import openai
from datasets import load_dataset

### 连接参数

请在你的 Astra 仪表盘上获取数据库凭据（[说明](https://docs.datastax.com/en/astra/astra-db-vector/)）：稍后你就需要提供它们。

示例值：

- API 端点：`https://01234567-89ab-cdef-0123-456789abcdef-us-east1.apps.astra.datastax.com`
- 令牌：`AstraCS:6gBhNmsk135...`

In [3]:
ASTRA_DB_API_ENDPOINT = input("Please enter your API Endpoint:")
ASTRA_DB_APPLICATION_TOKEN = getpass("Please enter your Token")

Please enter your API Endpoint: https://4f835778-ec78-42b0-9ae3-29e3cf45b596-us-east1.apps.astra.datastax.com
Please enter your Token ········


### 实例化 Astra DB 客户端

In [4]:
astra_db = AstraDB(
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    token=ASTRA_DB_APPLICATION_TOKEN,
)

## 创建向量集合

除了集合名称之外，唯一需要指定的参数是你要存储的向量的维度。其他参数，尤其是搜索时使用的相似度度量，都是可选的。

In [5]:
coll_name = "philosophers_astra_db"
collection = astra_db.create_collection(coll_name, dimension=1536)

## 连接 OpenAI

### 设置你的密钥

In [6]:
OPENAI_API_KEY = getpass("Please enter your OpenAI API Key: ")

Please enter your OpenAI API Key:  ········


### 嵌入的测试调用

快速检查一下如何为一组输入文本获取嵌入向量：

In [7]:
client = openai.OpenAI(api_key=OPENAI_API_KEY)
embedding_model_name = "text-embedding-3-small"

result = client.embeddings.create(
    input=[
        "This is a sentence",
        "A second sentence"
    ],
    model=embedding_model_name,
)

_注意：以上是 OpenAI v1.0+ 的语法。如果使用更早的版本，获取嵌入的代码会有所不同。_

In [8]:
print(f"len(result.data)              = {len(result.data)}")
print(f"result.data[1].embedding      = {str(result.data[1].embedding)[:55]}...")
print(f"len(result.data[1].embedding) = {len(result.data[1].embedding)}")

len(result.data)              = 2
result.data[1].embedding      = [-0.0108176339417696, 0.0013546717818826437, 0.00362232...
len(result.data[1].embedding) = 1536


## 将名言加载到向量存储中

获取包含名言的数据集。_（我们改编并扩充了[这个 Kaggle 数据集](https://www.kaggle.com/datasets/mertbozkurt5/quotes-by-philosophers)中的数据，可直接用于本演示。）_

In [9]:
philo_dataset = load_dataset("datastax/philosopher-quotes")["train"]

快速查看一下：

In [10]:
print("An example entry:")
print(philo_dataset[16])

An example entry:
{'author': 'aristotle', 'quote': 'Love well, be loved and do something of value.', 'tags': 'love;ethics'}


检查数据集大小：

In [11]:
author_count = Counter(entry["author"] for entry in philo_dataset)
print(f"Total: {len(philo_dataset)} quotes. By author:")
for author, count in author_count.most_common():
    print(f"    {author:<20}: {count} quotes")

Total: 450 quotes. By author:
    aristotle           : 50 quotes
    schopenhauer        : 50 quotes
    spinoza             : 50 quotes
    hegel               : 50 quotes
    freud               : 50 quotes
    nietzsche           : 50 quotes
    sartre              : 50 quotes
    plato               : 50 quotes
    kant                : 50 quotes


### 写入向量集合

你将计算名言的嵌入，并将其连同文本本身和之后要用到的元数据一起保存到向量存储中。

为了提高速度并减少调用次数，你将对 OpenAI 嵌入服务进行批量调用。

为了存储名言对象，你将使用集合的 `insert_many` 方法（每批调用一次）。在准备待插入的文档时，你可以选择合适的字段名——但请记住，嵌入向量必须放在固定的特殊字段 `$vector` 中。

In [12]:
BATCH_SIZE = 20

num_batches = ((len(philo_dataset) + BATCH_SIZE - 1) // BATCH_SIZE)

quotes_list = philo_dataset["quote"]
authors_list = philo_dataset["author"]
tags_list = philo_dataset["tags"]

print("Starting to store entries: ", end="")
for batch_i in range(num_batches):
    b_start = batch_i * BATCH_SIZE
    b_end = (batch_i + 1) * BATCH_SIZE
    # compute the embedding vectors for this batch
    b_emb_results = client.embeddings.create(
        input=quotes_list[b_start : b_end],
        model=embedding_model_name,
    )
    # prepare the documents for insertion
    b_docs = []
    for entry_idx, emb_result in zip(range(b_start, b_end), b_emb_results.data):
        if tags_list[entry_idx]:
            tags = {
                tag: True
                for tag in tags_list[entry_idx].split(";")
            }
        else:
            tags = {}
        b_docs.append({
            "quote": quotes_list[entry_idx],
            "$vector": emb_result.embedding,
            "author": authors_list[entry_idx],
            "tags": tags,
        })
    # write to the vector collection
    collection.insert_many(b_docs)
    print(f"[{len(b_docs)}]", end="")

print("\nFinished storing entries.")

Starting to store entries: [20][20][20][20][20][20][20][20][20][20][20][20][20][20][20][20][20][20][20][20][20][20][10]
Finished storing entries.


## 用例 1：**名言搜索引擎**

对于名言搜索功能，你需要先将输入名言转换为向量，然后用它查询存储（此外还要在搜索调用中处理可选的元数据）。

将搜索引擎功能封装到一个函数中，以便复用。其核心是集合的 `vector_find` 方法：

In [13]:
def find_quote_and_author(query_quote, n, author=None, tags=None):
    query_vector = client.embeddings.create(
        input=[query_quote],
        model=embedding_model_name,
    ).data[0].embedding
    filter_clause = {}
    if author:
        filter_clause["author"] = author
    if tags:
        filter_clause["tags"] = {}
        for tag in tags:
            filter_clause["tags"][tag] = True
    #
    results = collection.vector_find(
        query_vector,
        limit=n,
        filter=filter_clause,
        fields=["quote", "author"]
    )
    return [
        (result["quote"], result["author"])
        for result in results
    ]

### 测试搜索

只传入一条名言：

In [14]:
find_quote_and_author("We struggle all our life for nothing", 3)

[('Life to the great majority is only a constant struggle for mere existence, with the certainty of losing it at last.',
  'schopenhauer'),
 ('We give up leisure in order that we may have leisure, just as we go to war in order that we may have peace.',
  'aristotle'),
 ('Perhaps the gods are kind to us, by making life more disagreeable as we grow older. In the end death seems less intolerable than the manifold burdens we carry',
  'freud')]

限定作者的搜索：

In [15]:
find_quote_and_author("We struggle all our life for nothing", 2, author="nietzsche")

[('To live is to suffer, to survive is to find some meaning in the suffering.',
  'nietzsche'),
 ('What makes us heroic?--Confronting simultaneously our supreme suffering and our supreme hope.',
  'nietzsche')]

限定标签的搜索（从之前随名言一起保存的标签中选择）：

In [16]:
find_quote_and_author("We struggle all our life for nothing", 2, tags=["politics"])

[('He who seeks equality between unequals seeks an absurdity.', 'spinoza'),
 ('The people are that part of the state that does not know what it wants.',
  'hegel')]

### 剔除不相关的结果

向量相似度搜索通常会返回与查询最接近的向量，即使在没有更好结果时，这些结果可能有些不相关。

为了控制这个问题，你可以获取查询与每个结果之间实际的“相似度”，然后对其设置一个截断值，从而丢弃超出该阈值的结果。
正确调优这个阈值并不容易：这里我们只是给你演示一下方法。

为了直观感受其效果，可以尝试下面的查询，并调整所选名言和阈值来比较结果。请注意，相似度会作为每个结果文档中的特殊字段 `$similarity` 返回——默认会返回该字段，除非你向搜索方法传入 `include_similarity = False`。

_注意（写给喜欢数学的读者）：这个值是向量之间余弦差异**在零到一之间的重新缩放**，即两个向量的标量积除以两者范数的乘积。换句话说，方向相反的向量对应 0，平行的向量对应 +1。对于其他相似度度量（默认是余弦），请查看 `AstraDB.create_collection` 中的 `metric` 参数以及[关于可用取值的文档](https://docs.datastax.com/en/astra-serverless/docs/develop/dev-with-json.html#metric-types)。_

In [17]:
quote = "Animals are our equals."
# quote = "Be good."
# quote = "This teapot is strange."

metric_threshold = 0.92

quote_vector = client.embeddings.create(
    input=[quote],
    model=embedding_model_name,
).data[0].embedding

results_full = collection.vector_find(
    quote_vector,
    limit=8,
    fields=["quote"]
)
results = [res for res in results_full if res["$similarity"] >= metric_threshold]

print(f"{len(results)} quotes within the threshold:")
for idx, result in enumerate(results):
    print(f"    {idx}. [similarity={result['$similarity']:.3f}] \"{result['quote'][:70]}...\"")

3 quotes within the threshold:
    0. [similarity=0.927] "The assumption that animals are without rights, and the illusion that ..."
    1. [similarity=0.922] "Animals are in possession of themselves; their soul is in possession o..."
    2. [similarity=0.920] "At his best, man is the noblest of all animals; separated from law and..."


## 用例 2：**名言生成器**

对于这个任务，你需要 OpenAI 的另一个组件，即一个 LLM，来为我们生成名言（基于查询向量存储获得的输入）。

你还需要一个提示词模板，用于填充生成名言的 LLM 补全任务。

In [18]:
completion_model_name = "gpt-3.5-turbo"

generation_prompt_template = """"Generate a single short philosophical quote on the given topic,
similar in spirit and form to the provided actual example quotes.
Do not exceed 20-30 words in your quote.

REFERENCE TOPIC: "{topic}"

ACTUAL EXAMPLES:
{examples}
"""

与搜索一样，这个功能最好也封装成一个方便的函数（其内部会用到搜索）：

In [19]:
def generate_quote(topic, n=2, author=None, tags=None):
    quotes = find_quote_and_author(query_quote=topic, n=n, author=author, tags=tags)
    if quotes:
        prompt = generation_prompt_template.format(
            topic=topic,
            examples="\n".join(f"  - {quote[0]}" for quote in quotes),
        )
        # a little logging:
        print("** quotes found:")
        for q, a in quotes:
            print(f"**    - {q} ({a})")
        print("** end of logging")
        #
        response = client.chat.completions.create(
            model=completion_model_name,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
            max_tokens=320,
        )
        return response.choices[0].message.content.replace('"', '').strip()
    else:
        print("** no quotes found.")
        return None

_注意：与计算嵌入的情况类似，在 OpenAI v1.0 之前的版本中，调用 Chat Completion API 的代码会略有不同。_

#### 测试名言生成

只传入一段文本（一条“名言”，但实际上也可以只建议一个主题，因为它的向量嵌入仍会落在向量空间中的正确位置）：

In [20]:
q_topic = generate_quote("politics and virtue")
print("\nA new generated quote:")
print(q_topic)

** quotes found:
**    - Happiness is the reward of virtue. (aristotle)
**    - Our moral virtues benefit mainly other people; intellectual virtues, on the other hand, benefit primarily ourselves; therefore the former make us universally popular, the latter unpopular. (schopenhauer)
** end of logging

A new generated quote:
True politics lies in the virtuous pursuit of justice, for it is through virtue that we build a better world for all.


只从一位哲学家那里汲取灵感：

In [21]:
q_topic = generate_quote("animals", author="schopenhauer")
print("\nA new generated quote:")
print(q_topic)

** quotes found:
**    - Because Christian morality leaves animals out of account, they are at once outlawed in philosophical morals; they are mere 'things,' mere means to any ends whatsoever. They can therefore be used for vivisection, hunting, coursing, bullfights, and horse racing, and can be whipped to death as they struggle along with heavy carts of stone. Shame on such a morality that is worthy of pariahs, and that fails to recognize the eternal essence that exists in every living thing, and shines forth with inscrutable significance from all eyes that see the sun! (schopenhauer)
**    - The assumption that animals are without rights, and the illusion that our treatment of them has no moral significance, is a positively outrageous example of Western crudity and barbarity. Universal compassion is the only guarantee of morality. (schopenhauer)
** end of logging

A new generated quote:
Excluding animals from ethical consideration reveals a moral blindness that allows for their explo

## 清理

如果你想删除本演示使用的所有资源，请运行此单元（_警告：这将不可逆地删除集合及其数据！_）：

In [22]:
astra_db.delete_collection(coll_name)

{'status': {'ok': 1}}